# DiD-BCF — B1_strong_confounder (linearity_degree=3)

**Workstream B1 · canonical DiD (selection on unobservables)**

large Var(alpha) and strong corr(alpha, treatment)

Fits DiD-BCF on the `B1_strong_confounder` scenario at `linearity_degree=3` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.4 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "B1_strong_confounder",
    linearity_degree=3,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[B1_strong_confounder_lin_3] canonical DGP | N=(200,) | reps=100 | 100 fits | jobs=1


B1_strong_confounder: 100%|██████████| 100/100 [6:04:58<00:00, 218.99s/fit]

[B1_strong_confounder_lin_3] wrote 2000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_B1_strong_confounder_lin_3.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,canonical,B1_strong_confounder,3,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.284667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.854342
1,canonical,B1_strong_confounder,3,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.290000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.854342
2,canonical,B1_strong_confounder,3,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.464667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.854342
3,canonical,B1_strong_confounder,3,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.480667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.854342
4,canonical,B1_strong_confounder,3,200,0,ES,k=0,NaN,NaN,0.0,...,0.284667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.854342


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,canonical,B1_strong_confounder,3,200,ATT,ATT,power,-1.038446,-3.724181,0.17,...,0.778970,2.397718,1.316201,3.724181,0.98,0.0,1.586675,3.726330,0.165450,6.671340
1,canonical,B1_strong_confounder,3,200,ES,k=0,power,-1.078631,-3.759477,0.16,...,0.948087,2.221608,1.319556,3.759477,0.98,0.0,1.603124,3.760584,0.203447,31.952548
2,canonical,B1_strong_confounder,3,200,ES,k=1,power,-1.037707,-3.736613,0.16,...,0.959843,2.457077,1.329849,3.736613,0.98,0.0,1.595659,3.739197,0.201665,5.729786
3,canonical,B1_strong_confounder,3,200,ES,k=2,power,-1.025582,-3.713110,0.20,...,0.949470,2.649607,1.323563,3.713110,0.97,0.0,1.597993,3.716239,0.197509,5.338943
4,canonical,B1_strong_confounder,3,200,ES,k=3,power,-1.011866,-3.687526,0.18,...,0.941246,2.835387,1.298650,3.687526,0.97,0.0,1.563869,3.691365,0.200875,4.874835
5,canonical,B1_strong_confounder,3,200,GATT,g=4_t=4,power,-1.078631,-3.759477,0.16,...,0.948087,2.221608,1.319556,3.759477,0.98,0.0,1.603124,3.760584,0.203447,31.952548
6,canonical,B1_strong_confounder,3,200,GATT,g=4_t=5,power,-1.037707,-3.736613,0.16,...,0.959843,2.457077,1.329849,3.736613,0.98,0.0,1.595659,3.739197,0.201665,5.729786
7,canonical,B1_strong_confounder,3,200,GATT,g=4_t=6,power,-1.025582,-3.713110,0.20,...,0.949470,2.649607,1.323563,3.713110,0.97,0.0,1.597993,3.716239,0.197509,5.338943
8,canonical,B1_strong_confounder,3,200,GATT,g=4_t=7,power,-1.011866,-3.687526,0.18,...,0.941246,2.835387,1.298650,3.687526,0.97,0.0,1.563869,3.691365,0.200875,4.874835


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,canonical,B1_strong_confounder,3,200,corrected,100,401.76,1.664991,0.726547,1.474191,...,0.384965,0.186770,0.167765,0.123406,0.199372,0.142964,0.791985,0.102892,0.949661,0.128521
1,canonical,B1_strong_confounder,3,200,plain,100,401.76,3.826187,0.125034,3.724181,...,0.989654,0.023669,0.000000,0.000000,0.000000,0.000000,2.005182,0.107811,2.548049,0.138955
